# Semana 4 · Sesión 1: Símbolos y expresiones simbólicas

**Módulo 1**

## Objetivos de la sesión

1. Crear símbolos con `sp.symbols` y suposiciones explícitas, y explicar
   qué resultados cambian según las suposiciones.
2. Reconocer una expresión como un árbol de `Add`, `Mul` y `Pow`, y saber
   hasta dónde llega la evaluación automática de SymPy.
3. Distinguir la igualdad **estructural** (`==`) de la igualdad
   **matemática** (`simplify`, `equals`).

## Antes de empezar

Hoy empieza el Módulo 1 y, con él, la herramienta central del curso:
**SymPy**. Todo lo anterior sigue en su lugar —el notebook de la semana 1,
las clases de la semana 2, Git de la semana 3— pero a partir de aquí el
contenido es cómputo simbólico.

Si la siguiente celda falla, revisa
[`preparacion.md`](../preparacion/preparacion.md) antes de seguir.

El cambio es de fondo: hasta hoy Python calculaba **números**; a partir de
hoy va a manipular **fórmulas**.

In [ ]:
import sympy as sp

# Muestra las expresiones con notación matemática (MathJax) en lugar de
# texto plano. Se llama una sola vez, al principio del notebook.
sp.init_printing()

sp.__version__

## Un símbolo no es una variable de Python

En Python, un nombre es una etiqueta que apunta a un valor: si escribes
`x = 2`, entonces `x + 1` vale `3` de inmediato.

Un **símbolo** de SymPy es otra cosa: representa una incógnita matemática,
sin valor asignado. `x + 1` con `x` simbólico no se puede calcular, así que
SymPy no lo calcula — construye y guarda la expresión `x + 1`.

Ojo con que hay **dos nombres** en juego: el de la variable de Python (`x`,
a la izquierda del `=`) y el del símbolo (`"x"`, entre comillas), que es lo
que SymPy imprime. Nada obliga a que coincidan.

In [ ]:
x = sp.Symbol("x")

print(type(x))
display(x + 1)   # no vale nada todavía: es la expresión, no un número

# Variable de Python: posicion. Nombre del símbolo: q.
posicion = sp.Symbol("q")

# Dos símbolos son el mismo si coinciden su nombre y sus suposiciones,
# aunque los hayas construido por separado:
posicion == sp.Symbol("q")

## `sp.symbols`: varios de una vez

Declarar uno por uno se vuelve tedioso. `sp.symbols` acepta una cadena con
varios nombres y devuelve una tupla, que se desempaqueta igual que
cualquier tupla de Python.

In [ ]:
x, y, z = sp.symbols("x y z")

# También acepta rangos, cómodos para índices: q1, q2, q3
coordenadas = sp.symbols("q1:4")

x, y, z, coordenadas

## Suposiciones: la física que le declaras a SymPy

Un símbolo **sin suposiciones**, como `sp.Symbol("x")`, es lo más general posible: para
SymPy podría ser un número complejo. Eso lo obliga a ser **muy** cauteloso,
y por eso a veces se niega a simplificar algo que a ti te parece evidente.

Las **suposiciones** (*assumptions*) le dicen qué clase de objeto es cada
símbolo. En física casi siempre lo sabemos: una masa es positiva, un número
cuántico es entero, una coordenada es real.

| Suposición | Se escribe | Ejemplo en física |
|---|---|---|
| Real | `real=True` | una coordenada, un tiempo |
| Positivo | `positive=True` | una masa, una temperatura absoluta, $c$ |
| No negativo | `nonnegative=True` | una distancia, una probabilidad |
| Entero | `integer=True` | un número cuántico $n$ |
| No conmutativo | `commutative=False` | un operador, una matriz |

Declararlas no es cosmético: cambia el resultado.

In [ ]:
generico = sp.Symbol("v")                   # podría ser complejo
positivo = sp.Symbol("v", positive=True)    # sabemos que v > 0

# La raíz de un cuadrado solo es el número original si no es negativo.
sp.sqrt(generico**2), sp.sqrt(positivo**2)

El primero se queda como $\sqrt{v^2}$ porque SymPy no puede descartar que
$v$ sea negativa: si lo fuera, la respuesta sería $-v$. El segundo se
simplifica a $v$ porque se lo dijimos.

**Regla práctica del curso:** declara siempre las suposiciones que el
problema físico ya te garantiza. Cuando una simplificación "no funcione",
lo primero que hay que revisar es si a los símbolos les faltan suposiciones.

## Consultar suposiciones: la lógica de tres valores

SymPy responde a `.is_positive`, `.is_real` y compañía, pero no con dos
respuestas sino con **tres**: `True`, `False` y `None`.

`None` no significa "no". Significa **"no lo sé"**: con la información
disponible, SymPy no puede decidirlo. Confundir `None` con `False` es de
los errores más caros al empezar.

Fíjate además en que SymPy **deduce**: no vamos a decir en ningún lado que
`m` sea real, pero de `positive=True` se sigue que lo es. Las suposiciones
son un sistema de inferencia, no una lista de etiquetas sueltas.

In [ ]:
n = sp.Symbol("n", integer=True)
m = sp.Symbol("m", positive=True)

print("m.is_positive :", m.is_positive)   # True: se lo dijimos
print("m.is_real     :", m.is_real)       # True: se deduce de positivo
print("n.is_real     :", n.is_real)       # True: todo entero es real
print("n.is_positive :", n.is_positive)   # None: entero, pero ¿de qué signo?
print("x.is_positive :", x.is_positive)   # None: no sabemos nada de x

## TODO en clase 1

La energía cinética de una partícula es $E = \tfrac{1}{2} m v^2$, y al
despejar la rapidez queda

$$v = \sqrt{\frac{2E}{m}}$$

1. Declara `energia` y `masa` con las suposiciones que el problema físico
   garantiza (las dos cantidades son positivas) y construye la expresión de
   la rapidez.
2. Repite la construcción con símbolos **sin** suposiciones y compara: la
   misma fórmula, distinto resultado.

In [ ]:
# TODO en clase: declara los símbolos con suposiciones y construye la rapidez
energia = ...
masa = ...

rapidez = ...

## Una expresión es un árbol

Cuando operas símbolos, SymPy no ejecuta la cuenta: **construye un objeto**
que representa la operación. Y es un objeto de una clase corriente, de las
que ya sabes leer desde la semana 2.

- Una suma es un `Add`.
- Un producto es un `Mul`.
- Una potencia es un `Pow`.

Como cada operando puede ser a su vez otra operación, lo que queda es un
**árbol**. `sp.srepr` lo imprime tal cual, sin el maquillaje matemático.

In [ ]:
expresion = x**2 + 2*x + 1

print(type(expresion))
print(sp.srepr(expresion))

expresion

Léelo de fuera hacia dentro: la expresión completa es un `Add` de tres
cosas — un `Pow` (la $x^2$), un `Mul` (el $2x$) y un `Integer` (el $1$).
Hasta los números literales se envuelven en objetos de SymPy: `Integer(1)`,
no el `1` de Python.

Esa estructura es la razón de ser del cómputo simbólico. Manipular una
fórmula es recorrer y reescribir este árbol, que es justo lo que hacen
`expand`, `factor` y compañía en la sesión 2.

## Evaluación automática: lo barato se hace solo

SymPy no espera a que le pidas simplificar para hacer lo obvio. Al
**construir** la expresión ya aplica las reglas baratas y siempre válidas:
juntar términos idénticos, ordenar, absorber ceros y unos.

Lo caro —expandir, factorizar, identidades trigonométricas— no lo hace
solo: eso lo pides tú, y es el tema de la sesión 2. Si necesitas congelar
una expresión sin que la toque, los constructores aceptan `evaluate=False`.

In [ ]:
display(x + x)        # 2*x  : términos idénticos se juntan solos
display(x * x)        # x**2 : también
display(x + y)        # x + y: nada que juntar, se queda como está
display((x + 1)**2)   # no se expande solo: expandir es caro

display(sp.Add(x, x, evaluate=False))   # x + x, congelado tal cual

## Depuración en vivo: el error número uno con SymPy

Es tentador escribir un `if` sobre un símbolo. Antes de ejecutar la celda,
predice qué va a pasar:

In [ ]:
try:
    if x > 0:
        print("x es positiva")
except TypeError as error:
    print("TypeError:", error)

`x > 0` **no** es `True` ni `False`: es otra expresión simbólica —la
desigualdad $x > 0$— y SymPy se niega a inventarle un valor de verdad,
porque depende de una $x$ que no conoce.

La salida no es forzar la comparación, sino preguntar por la **suposición**:

```python
if x.is_positive:      # True, False o None
    ...
```

Y si de verdad necesitas que $x$ sea positiva, decláralo al crear el
símbolo. Es la idea de siempre: la información física va en las
suposiciones, no en un `if`.

## Otro tropiezo clásico: los flotantes se contagian

SymPy trabaja con números **exactos**. Un `1/3` de Python, en cambio, es un
flotante, y en cuanto entra en una expresión contamina todo lo que toca: el
resultado deja de ser exacto para siempre. Lo mismo vale para `math.pi`
frente a `sp.pi`.

In [ ]:
display((1 / 3) * x)              # 0.333...*x : ya perdimos la exactitud
display(sp.Rational(1, 3) * x)    # x/3        : exacto
display(sp.Integer(1) / 3 * x)    # x/3        : también exacto

## TODO en clase 2

El periodo de un péndulo simple es

$$T = 2\pi \sqrt{\frac{L}{g}}$$

1. Declara `longitud` y `gravedad` como símbolos positivos.
2. Construye la expresión del periodo, con `sp.pi` (no `math.pi`).
3. Imprime su estructura con `sp.srepr` y busca en el árbol los exponentes
   `Rational(1, 2)` y `Rational(-1, 2)`. ¿De dónde sale cada uno? Pista: en
   el árbol no existen ni la raíz ni la división — las dos son potencias.

In [ ]:
# TODO en clase: construye el periodo del péndulo y mira su árbol
longitud = ...
gravedad = ...

periodo = ...

## `==` compara estructura, no matemáticas

Esto es lo más importante de la sesión.

En SymPy, `a == b` pregunta si las dos expresiones son **el mismo árbol**,
no si son matemáticamente iguales. Por eso $(x+1)^2$ y $x^2 + 2x + 1$
resultan distintas: valen lo mismo para toda $x$, pero están escritas de
otra forma.

Para preguntar por igualdad **matemática** hay dos caminos:

| Quieres | Escribe | Devuelve |
|---|---|---|
| Comparar estructura | `a == b` | `True` / `False` |
| Comparar valor (recomendado) | `sp.simplify(a - b) == 0` | `True` / `False` |
| Comparar valor (simbólico y numérico) | `a.equals(b)` | `True` / `False` / `None` |
| Escribir la ecuación $a = b$ | `sp.Eq(a, b)` | una expresión simbólica |

La receta `sp.simplify(a - b) == 0` es la que usa el autograding de tus
tareas: si la diferencia se simplifica a cero, las dos expresiones son la
misma. Por eso no importa en qué forma escribas tu respuesta.

In [ ]:
a = (x + 1)**2
b = x**2 + 2*x + 1

print("a == b             :", a == b)
print("simplify(a - b) == 0:", sp.simplify(a - b) == 0)
print("a.equals(b)        :", a.equals(b))

# Y si lo que quieres es la ecuación como objeto, no una comparación:
sp.Eq(a, b)

## TODO en clase 3

Dos formas de la energía de un oscilador armónico que aparecen en libros
distintos:

$$E_1 = \tfrac{1}{2} k A^2 \sin^2(\omega t) + \tfrac{1}{2} k A^2 \cos^2(\omega t)
\qquad
E_2 = \tfrac{1}{2} k A^2$$

1. Declara `k`, `amplitud` y `omega` como positivos, y `tiempo` como real.
2. Escribe las dos expresiones.
3. Comprueba que `energia_1 == energia_2` da `False`, y que las dos recetas
   de igualdad matemática dan `True`.

Cuidado con la fracción: `1/2` es un flotante, `sp.Rational(1, 2)` no.

In [ ]:
# TODO en clase: escribe las dos formas de la energía y compáralas
k = ...
amplitud = ...
omega = ...
tiempo = ...

energia_1 = ...
energia_2 = ...

## Resumen

Hoy dimos el paso de calcular números a manipular fórmulas. Un símbolo es
un objeto sin valor; las **suposiciones** son la forma de meterle la física
al problema, y sin ellas SymPy se niega —con razón— a simplificar. Una
expresión es un **árbol** de `Add`, `Mul` y `Pow`, y SymPy lo evalúa solo
hasta donde sale barato.

También vimos las dos trampas que más tiempo cuestan: usar un símbolo
dentro de un `if`, y dejar entrar un flotante de Python donde tocaba un
número exacto.

Y lo esencial: `==` compara **estructura**. La igualdad matemática se
pregunta con `sp.simplify(a - b) == 0`.

**Próxima sesión — Semana 4, sesión 2:** reescribir esas expresiones a
voluntad (`expand`, `factor`, `collect`, `simplify`), sustituir con `subs`
y bajar por fin al número con `evalf` y `lambdify`, para poder graficar.